# Growth-rate analysis by COVID-19 wave
**Question:** Does the per-capita weekly growth rate decrease approximately linearly with cumulative cases within each wave, as the logistic model predicts?

The growth proxy is weekly new cases divided by cumulative cases since wave start. It is noisy and not the instantaneous derivative, but it gives a useful descriptive comparison. We report OLS slopes and R², without treating them as causal evidence.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import linregress
ROOT=Path.cwd()
if not (ROOT/"data/processed/us_covid_all_waves.csv").exists(): ROOT=ROOT.parent
DATA=ROOT/"data/processed/us_covid_all_waves.csv"
df=pd.read_csv(DATA,parse_dates=["week_ending"])
required={"wave","new_cases","cumulative_cases_in_wave","week_ending"}
missing=required-set(df.columns)
if missing: raise ValueError(f"Missing columns: {missing}")
df=df.sort_values(["wave","week_ending"]).copy()
df["growth_proxy"]=df["new_cases"]/df["cumulative_cases_in_wave"].replace(0,np.nan)
df=df.replace([np.inf,-np.inf],np.nan).dropna(subset=["growth_proxy"])
df=df[df["cumulative_cases_in_wave"]>0]
df.head()

## 1. Wave-wise scatterplots and linear fits

In [ ]:
summary=[]
waves=sorted(df.wave.unique())
fig,axes=plt.subplots(len(waves),1,figsize=(10,3.6*len(waves)),squeeze=False)
for ax,w in zip(axes[:,0],waves):
    sub=df[df.wave==w]
    if len(sub)<3: continue
    fit=linregress(sub.cumulative_cases_in_wave,sub.growth_proxy)
    summary.append({"wave":int(w),"weeks":len(sub),"slope":fit.slope,"intercept":fit.intercept,"r_squared":fit.rvalue**2,"p_value":fit.pvalue})
    x=np.linspace(sub.cumulative_cases_in_wave.min(),sub.cumulative_cases_in_wave.max(),150)
    ax.scatter(sub.cumulative_cases_in_wave,sub.growth_proxy,s=20,alpha=.65)
    ax.plot(x,fit.intercept+fit.slope*x,label=f"OLS R²={fit.rvalue**2:.3f}, slope={fit.slope:.2e}")
    ax.set(title=f"Wave {w}: growth proxy vs. cumulative cases",xlabel="Cumulative cases in wave",ylabel="Weekly new / cumulative")
    ax.legend(); ax.grid(alpha=.25)
plt.tight_layout()

In [ ]:
growth_summary=pd.DataFrame(summary)
growth_summary

## 2. Results and interpretation
A logistic model implies a declining per-capita growth rate as cumulative cases approach carrying capacity. A negative slope is consistent with that qualitative pattern, but a weak fit or positive slope indicates that a single logistic curve may not capture the wave's changing dynamics. Reporting changes, variants, behavior, immunity, and wave segmentation can all distort the proxy.

In [ ]:
display(growth_summary.sort_values("wave"))
out=ROOT/"results/growth_rate_by_wave.csv"
out.parent.mkdir(parents=True,exist_ok=True)
growth_summary.to_csv(out,index=False)
print("Saved:",out)